# **Silver Layer - Transformations**

In [0]:
# importing libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

## **Dynamic Optimization**
This is manditory for accessing the python classes into notebook for performing common transformations 

In [0]:
# import libraries
import os
import sys

# add CURRENT SOURCE PATH to SYSTEM PATH
current_dir = os.getcwd()
sys.path.append(current_dir)


### **For Customers**

In [0]:
# reading data for customers
df_customers = spark.read.table('databricks_pyspark_dbt_project_01.bronze_schema.customers_data')

In [0]:
# transformation : get domain name from email col
df_customers = df_customers.withColumn('domain', split(col('email'), '@')[1])


In [0]:
# transformation : replace all the other symbols from phone_number col   
df_customers =  df_customers.withColumn('phone_number', regexp_replace('phone_number', r'[^0-9]', ""))

In [0]:
# transformation : add a col 'full_name' by concating col 'first_name' & 'last_name' AND drop after concating drop col 'first_name' & 'last_name'
df_customers = df_customers.withColumn('full_name', concat_ws(" ", col('first_name'), col('last_name')))
df_customers = df_customers.drop('first_name', 'last_name')

In [0]:
import importlib
import utlis.custom_utlis

# force python to reload module
importlib.reload(utlis.custom_utlis)

# import class transformation
from utlis.custom_utlis import transformations

In [0]:
# create obj and provide parameters for dedup
customer_object = transformations()
customer_transformation = customer_object.dedup(df_customers, ['customer_id'], 'last_updated_timestamp')

In [0]:
# provide parameter for processing_timestamp
df_customers = customer_object.processing_timestamp(df_customers)

In [0]:
# TRANSFORMATION - UPSERT 
# if table not exist then write data and saveAsTable
if not spark.catalog.tableExists('databricks_pyspark_dbt_project_01.silver_schema.customers_data'):
    df_customers.write\
                .mode("append")\
                .format("delta")\
                .saveAsTable("databricks_pyspark_dbt_project_01.silver_schema.customers_data")
    
# if table exist then upsert data
else:
    customer_object.upsert(df_customers, ['customer_id'], 'customers_data', 'last_updated_timestamp')

### **For Drivers**

In [0]:
# reading data
df_drivers = spark.read.table('databricks_pyspark_dbt_project_01.bronze_schema.drivers_data')

In [0]:
# transformation : replace all the other symbols from phone_number col   
df_drivers =  df_drivers.withColumn('phone_number', regexp_replace('phone_number', r'[^0-9]', ""))

# transformation : add a col 'full_name' by concating col 'first_name' & 'last_name' AND drop after concating drop col 'first_name' & 'last_name'
df_drivers = df_drivers.withColumn('full_name', concat_ws(" ", col('first_name'), col('last_name')))
df_drivers = df_drivers.drop('first_name', 'last_name')

In [0]:
# create obj and provide parameters for dedup
driver_object = transformations()
driver_transformation = driver_object.dedup(df_drivers, ['driver_id'], 'last_updated_timestamp')

# provide parameter for processing_timestamp
df_drivers = driver_object.processing_timestamp(df_drivers)

In [0]:
# TRANSFORMATION - UPSERT 
# if table not exist then write data and saveAsTable
if not spark.catalog.tableExists('databricks_pyspark_dbt_project_01.silver_schema.drivers_data'):
    df_drivers.write\
                .mode("append")\
                .format("delta")\
                .saveAsTable("databricks_pyspark_dbt_project_01.silver_schema.drivers_data")
    
# if table exist then upsert data
else:
    driver_object.upsert(df_drivers, ['driver_id'], 'drivers_data', 'last_updated_timestamp')

### **For Locations**

In [0]:
# reading data for locations
df_locations = spark.read.table('databricks_pyspark_dbt_project_01.bronze_schema.locations_data')

In [0]:
df_locations.display()

In [0]:
# create obj and provide parameters for dedup
location_object = transformations()
location_transformation = location_object.dedup(df_locations, ['location_id'], 'last_updated_timestamp')

# provide parameter for processing_timestamp
df_locations = location_object.processing_timestamp(df_locations)

In [0]:
# TRANSFORMATION - UPSERT 
# if table not exist then write data and saveAsTable
if not spark.catalog.tableExists('databricks_pyspark_dbt_project_01.silver_schema.locations_data'):
    df_locations.write\
                .mode("append")\
                .format("delta")\
                .saveAsTable("databricks_pyspark_dbt_project_01.silver_schema.locations_data")
    
# if table exist then upsert data
else:
    location_object.upsert(df_locations, ['location_id'], 'locations_data', 'last_updated_timestamp')

### **For Payments**

In [0]:
# reading payments data
df_payments = spark.read.table('databricks_pyspark_dbt_project_01.bronze_schema.payments_data')

In [0]:
# transformation : create a new col *online-payment-status* to identify online payments status
df_payments = df_payments.withColumn('online-payment-status',
                    when((col('payment_method') == 'Card') & (col('payment_status') == 'Success'), "Online-Success")
                    .when((col('payment_method') == 'Card') & (col('payment_status') == 'Failed'), "Online-Failed")
                    .when((col('payment_method') == 'Card') & (col('payment_status') == 'Pending'), "Online-Pending")
                    .otherwise('Offline')
                )


In [0]:
# create obj and provide parameters for dedup
payment_object = transformations()
payment_transformation = payment_object.dedup(df_payments, ['payment_id'], 'last_updated_timestamp')

# provide parameter for processing_timestamp
df_payments = payment_object.processing_timestamp(df_payments)

In [0]:
# TRANSFORMATION - UPSERT 
# if table not exist then write data and saveAsTable
if not spark.catalog.tableExists('databricks_pyspark_dbt_project_01.silver_schema.payments_data'):
    df_payments.write\
                .mode("append")\
                .format("delta")\
                .saveAsTable("databricks_pyspark_dbt_project_01.silver_schema.payments_data")
    
# if table exist then upsert data
else:
    payment_object.upsert(df_payments, ['payment_id'], 'payments_data', 'last_updated_timestamp')

### **For Trips**

In [0]:
# reading trips data
df_vehicles = spark.read.table('databricks_pyspark_dbt_project_01.bronze_schema.vehicles_data')

In [0]:
# transformation : convert model col into UPPERCASE
df_vehicles = df_vehicles.withColumn("model", upper(col("model")))

In [0]:
# create obj and provide parameters for dedup
vehicle_object = transformations()
vehicle_transformation = vehicle_object.dedup(df_vehicles, ['vehicle_id'], 'last_updated_timestamp')

# provide parameter for processing_timestamp
df_vehicles = vehicle_object.processing_timestamp(df_vehicles)

In [0]:
# TRANSFORMATION - UPSERT 
# if table not exist then write data and saveAsTable
if not spark.catalog.tableExists('databricks_pyspark_dbt_project_01.silver_schema.vehicles_data'):
    df_vehicles.write\
                .mode("append")\
                .format("delta")\
                .saveAsTable("databricks_pyspark_dbt_project_01.silver_schema.vehicles_data")
    
# if table exist then upsert data
else:
    vehicle_object.upsert(df_vehicles, ['payment_id'], 'vehicles_data', 'last_updated_timestamp')

### **For Trips**

In [0]:
# start integrating DBT tool - pause the project untill I learn DBT